In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm


/home/ashish/anaconda3/envs/realesrgan/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
# ============================================================
# CONFIG
# ============================================================
DATA_DIR_NORMAL = "./Dataset/Original"
DATA_DIR_DENOISED = "./Dataset/Denoised"
IMG_SIZE = 128
BATCH_SIZE = 32
EPOCHS = 50
LR = 1e-3
OUTPUT_DIR = "./trained_models_custom_cnn"
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")

✅ Using device: cuda


In [10]:
# ============================================================
# DATA PIPELINE
# ============================================================
transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

def load_datasets(data_dir):
    dataset = datasets.ImageFolder(data_dir, transform=transform)
    train_size = int(0.8 * len(dataset))
    val_size = len(dataset) - train_size
    train_ds, val_ds = random_split(dataset, [train_size, val_size])
    dataloaders = {
        'train': DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=4),
        'val': DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
    }
    return dataloaders

In [11]:
# ============================================================
# MODEL DEFINITIONS
# ============================================================

class CNN_A(nn.Module):
    """Simple CNN with L2 regularization"""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * (IMG_SIZE // 4) * (IMG_SIZE // 4), 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

In [12]:
class CNN_B(nn.Module):
    """Deeper CNN with dropout"""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(128 * (IMG_SIZE // 4) * (IMG_SIZE // 4), 256),
            nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256, 1)
        )
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)

In [13]:
class CNN_C(nn.Module):
    """Compact CNN with pruning-friendly sparse layers"""
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * (IMG_SIZE // 4) * (IMG_SIZE // 4), 128),
            nn.ReLU(),
            nn.Linear(128, 1)
        )
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)
        return self.classifier(x)


In [14]:
# ============================================================
# TRAINING FUNCTION
# ============================================================
def train_model(model, dataloaders, model_name, data_type):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-4)  # L2 regularization
    best_acc = 0.0

    for epoch in range(EPOCHS):
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss, running_corrects = 0.0, 0
            for inputs, labels in tqdm(dataloaders[phase], desc=f"{phase} - {model_name} ({data_type})"):
                inputs, labels = inputs.to(device), labels.to(device).float()
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs).squeeze()
                    preds = torch.round(torch.sigmoid(outputs))
                    loss = criterion(outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels)

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_acc = running_corrects.double() / len(dataloaders[phase].dataset)
            print(f"Epoch [{epoch+1}/{EPOCHS}] {phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}")

            # Early stopping on overfitting
            if phase == 'val' and epoch_acc < best_acc - 0.01:
                print("🛑 Early stopping: overfitting detected.")
                return best_acc

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, f"{model_name}_{data_type}_best.pth"))

    return best_acc


In [15]:
# ============================================================
# MAIN EXPERIMENT
# ============================================================
MODELS = {
    "CNN_A": CNN_A(),
    "CNN_B": CNN_B(),
    "CNN_C": CNN_C()
}

results = {}

for model_name, model in MODELS.items():
    for data_type, data_dir in [("normal", DATA_DIR_NORMAL), ("denoised", DATA_DIR_DENOISED)]:
        dataloaders = load_datasets(data_dir)
        model = model.to(device)
        print(f"\n🚀 Training {model_name} on {data_type} dataset...")
        acc = train_model(model, dataloaders, model_name, data_type)
        results[f"{model_name}_{data_type}"] = acc.item()

print("\n✅ Final Accuracy Comparison:")
for k, v in results.items():
    print(f"{k}: {v:.4f}")


🚀 Training CNN_A on normal dataset...


train - CNN_A (normal): 100%|██████████| 46/46 [00:02<00:00, 21.21it/s]


Epoch [1/50] train Loss: 0.7745 Acc: 0.5546


val - CNN_A (normal): 100%|██████████| 12/12 [00:00<00:00, 25.11it/s]


Epoch [1/50] val Loss: 0.6866 Acc: 0.4808


...

Epoch [3/50] val Loss: 0.2546 Acc: 0.9124


train - CNN_C (denoised): 100%|██████████| 18/18 [00:05<00:00,  3.08it/s]


Epoch [4/50] train Loss: 0.2629 Acc: 0.9066


val - CNN_C (denoised): 100%|██████████| 5/5 [00:01<00:00,  3.62it/s]

Epoch [4/50] val Loss: 0.2560 Acc: 0.8832
🛑 Early stopping: overfitting detected.

✅ Final Accuracy Comparison:
CNN_A_normal: 0.7555
CNN_A_denoised: 0.9416
CNN_B_normal: 0.7390
CNN_B_denoised: 0.8613
CNN_C_normal: 0.7802
CNN_C_denoised: 0.9124
